# TIR Super-Resolution + IR Colorization Dataset Pipeline

This notebook contains the full reusable workflow for the Landsat 9 TIR super-resolution and colorization dataset.

It is designed so you can continue the work even if Colab restarts, because all important files are stored in Google Drive.

## What this notebook does

1. Verifies existing exported `processed100m_*.tif` files in Drive.
2. Safely exports new processed Landsat 9 scenes without duplicate indices.
3. Copies processed scenes to local Colab storage for faster processing.
4. Extracts repo-compatible `.npy` patches:
   - SR: `TIR_200m 256×256 → TIR_100m 512×512`
   - Colorization: `TIR_100m 256×256 → RGB_100m 3×256×256`
5. Counts and visualizes generated samples.
6. Provides PyTorch dataloaders for model development.
7. Saves and zips the dataset for IDE/GitHub workflow.

## Important rule

Never regenerate patches by appending blindly. Always rebuild `dataset_current` from all available `processed100m_*.tif` files to avoid duplicate `.npy` patches.

## 0. Install/import dependencies

Run this first in a fresh Colab session.

In [ ]:
# Usually preinstalled in Colab, but safe to run if needed.
# !pip install rasterio earthengine-api

import os
import re
import csv
import glob
import shutil
import random
from collections import Counter

import numpy as np
import matplotlib.pyplot as plt

import rasterio

print("Imports ready")

## 1. Mount Google Drive and set paths

Drive is permanent storage. `/content/` is temporary and disappears after Colab restarts.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

DRIVE_ROOT = "/content/drive/MyDrive/landsat_india_200"
LOCAL_ROOT = "/content/landsat_india_200_work"
LOCAL_SCENE_DIR = os.path.join(LOCAL_ROOT, "processed_100m_scenes")

os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(LOCAL_SCENE_DIR, exist_ok=True)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("LOCAL_ROOT:", LOCAL_ROOT)
print("LOCAL_SCENE_DIR:", LOCAL_SCENE_DIR)

## 2. Check existing exported scenes and redundancy

Use this after every restart before exporting more scenes. It checks:
- which `processed100m_*.tif` files exist
- whether any scene index is duplicated
- which indices are missing from an expected range

In [ ]:
def check_existing_processed_scenes(expected_end=None):
    scene_files = sorted(glob.glob(os.path.join(DRIVE_ROOT, "processed100m_*.tif")))

    indices = []
    for f in scene_files:
        base = os.path.basename(f)
        m = re.match(r"processed100m_(\d{3})_", base)
        if m:
            indices.append(int(m.group(1)))

    indices = sorted(indices)
    counts = Counter(indices)
    duplicates = sorted([idx for idx, c in counts.items() if c > 1])

    print("Total files:", len(scene_files))
    print("Found indices:", indices)
    print("Duplicate indices:", duplicates)

    if expected_end is not None:
        expected = list(range(0, expected_end))
        missing = sorted(set(expected) - set(indices))
        print(f"Missing from 0 to {expected_end-1}:", missing)

    return scene_files, indices, duplicates

scene_files, indices, duplicates = check_existing_processed_scenes(expected_end=50)

## 3. Earth Engine setup

Only run this section when you need to export more processed 100m scene files.

Current project ID used in this work:

```text
tir-project-499614
```

In [ ]:
import ee

ee.Authenticate(force=True)
ee.Initialize(project="tir-project-499614")

print("Earth Engine initialized")

## 4. Build good Landsat 9 scene collection

This uses:
- Landsat 9 Collection 2 Level 2
- India boundary
- date range: 2022–2025
- cloud cover < 10%
- centroid inside India, so scenes do not only barely touch India

In [ ]:
N_CANDIDATES = 300

india_fc = (
    ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017")
    .filter(ee.Filter.eq("country_na", "India"))
)

india = india_fc.geometry()

def add_centroid_inside_india(img):
    centroid = img.geometry().centroid(100)
    inside = india.contains(centroid, ee.ErrorMargin(100))
    return img.set("centroid_inside_india", inside)

collection_good = (
    ee.ImageCollection("LANDSAT/LC09/C02/T1_L2")
    .filterBounds(india)
    .filterDate("2022-01-01", "2025-01-01")
    .filter(ee.Filter.lt("CLOUD_COVER", 10))
    .map(add_centroid_inside_india)
    .filter(ee.Filter.eq("centroid_inside_india", True))
    .sort("CLOUD_COVER")
    .limit(N_CANDIDATES)
)

print("Good candidate scenes:", collection_good.size().getInfo())

scene_list = collection_good.toList(N_CANDIDATES)
scene_ids = collection_good.aggregate_array("LANDSAT_PRODUCT_ID").getInfo()
clouds = collection_good.aggregate_array("CLOUD_COVER").getInfo()

print("Scene IDs loaded:", len(scene_ids))
print(scene_ids[:10])

## 5. Landsat preprocessing function

This creates a cleaned 100m 4-band image:

```text
Band 1 = R
Band 2 = G
Band 3 = B
Band 4 = TIR
```

Processing done:
- QA cloud/fill/shadow/snow masking
- Landsat Level-2 scale factors
- valid reflectance range filtering
- valid temperature range filtering
- resampling to 100m

In [ ]:
def preprocess_landsat(img):
    qa = img.select("QA_PIXEL")

    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)  # fill
        .And(qa.bitwiseAnd(1 << 1).eq(0))  # dilated cloud
        .And(qa.bitwiseAnd(1 << 2).eq(0))  # cirrus
        .And(qa.bitwiseAnd(1 << 3).eq(0))  # cloud
        .And(qa.bitwiseAnd(1 << 4).eq(0))  # cloud shadow
        .And(qa.bitwiseAnd(1 << 5).eq(0))  # snow
    )

    blue = img.select("SR_B2").multiply(0.0000275).add(-0.2).rename("B")
    green = img.select("SR_B3").multiply(0.0000275).add(-0.2).rename("G")
    red = img.select("SR_B4").multiply(0.0000275).add(-0.2).rename("R")
    tir = img.select("ST_B10").multiply(0.00341802).add(149.0).rename("TIR")

    out = red.addBands(green).addBands(blue).addBands(tir)
    out = out.updateMask(mask)

    valid = (
        out.select("R").gte(0).And(out.select("R").lte(1))
        .And(out.select("G").gte(0)).And(out.select("G").lte(1))
        .And(out.select("B").gte(0)).And(out.select("B").lte(1))
        .And(out.select("TIR").gte(250)).And(out.select("TIR").lte(350))
    )

    out = out.updateMask(valid)

    proj = img.select("ST_B10").projection()

    out100 = out.resample("bilinear").reproject(crs=proj, scale=100)

    return out100

## 6. Safe batch export to Drive

Change `TARGET_START` and `TARGET_END` to export the next batch.

Examples:

```python
TARGET_START = 50
TARGET_END = 70
```

The exporter checks existing indices and skips files that are already in Drive, so it avoids redundancy after Colab restarts.

In [ ]:
DRIVE_FOLDER = "landsat_india_200"

TARGET_START = 50
TARGET_END = 70

def safe_export_batch(target_start, target_end):
    existing_files = sorted(glob.glob(os.path.join(DRIVE_ROOT, "processed100m_*.tif")))
    existing_indices = set()

    for f in existing_files:
        base = os.path.basename(f)
        m = re.match(r"processed100m_(\d{3})_", base)
        if m:
            existing_indices.add(int(m.group(1)))

    tasks = []

    for i in range(target_start, target_end):
        if i in existing_indices:
            print("Already exists, skipping:", i)
            continue

        product_id = scene_ids[i]
        img_raw = ee.Image(scene_list.get(i))
        img100 = preprocess_landsat(img_raw)

        task = ee.batch.Export.image.toDrive(
            image=img100,
            description=f"processed100m_{i:03d}",
            folder=DRIVE_FOLDER,
            fileNamePrefix=f"processed100m_{i:03d}_{product_id}",
            region=img_raw.geometry(),
            scale=100,
            maxPixels=1e13,
            fileFormat="GeoTIFF"
        )

        task.start()
        tasks.append(task)
        print("Started:", i, product_id)

    print("New tasks started:", len(tasks))
    return tasks

# Uncomment when you want to export the next batch.
# tasks = safe_export_batch(TARGET_START, TARGET_END)

## 7. Check Earth Engine task status

Run this after starting export tasks. Because `tasks` is a runtime variable, it disappears after Colab restarts. If Colab restarted, check files in Drive instead using section 2.

In [ ]:
def check_tasks(tasks):
    completed = 0
    running = 0
    ready = 0
    failed = 0

    for t in tasks:
        status = t.status()
        state = status["state"]

        if state == "COMPLETED":
            completed += 1
        elif state == "RUNNING":
            running += 1
        elif state == "READY":
            ready += 1
        elif state == "FAILED":
            failed += 1
            print("FAILED TASK:")
            print(status)

    print("COMPLETED:", completed)
    print("RUNNING:", running)
    print("READY:", ready)
    print("FAILED:", failed)

# Example:
# check_tasks(tasks)

## 8. Copy processed scenes from Drive to local Colab storage

Patch extraction is much faster from `/content/` than directly from mounted Google Drive.

In [ ]:
def copy_processed_scenes_to_local():
    os.makedirs(LOCAL_SCENE_DIR, exist_ok=True)

    scene_files = sorted(glob.glob(os.path.join(DRIVE_ROOT, "processed100m_*.tif")))
    print("Scenes in Drive:", len(scene_files))

    for f in scene_files:
        dst = os.path.join(LOCAL_SCENE_DIR, os.path.basename(f))
        if not os.path.exists(dst):
            shutil.copy2(f, dst)

    local_scene_files = sorted(glob.glob(os.path.join(LOCAL_SCENE_DIR, "processed100m_*.tif")))
    print("Scenes locally:", len(local_scene_files))
    return local_scene_files

local_scene_files = copy_processed_scenes_to_local()

## 9. Repo-compatible patch extraction

This regenerates `dataset_current` from scratch using all local processed scenes.

Output format:

```text
dataset_current/
  sr/
    train/tir_200m/*.npy    # 256×256
    train/tir_100m/*.npy    # 512×512
    val/...
    test/...

  colorization/
    train/tir_100m/*.npy    # 256×256
    train/rgb_100m/*.npy    # 3×256×256
    val/...
    test/...

  metadata/
    patch_metadata.csv
    scene_summary.csv
```

In [ ]:
# Delete old local dataset_current before regenerating.
OUT_ROOT = os.path.join(LOCAL_ROOT, "dataset_current")
META_DIR = os.path.join(OUT_ROOT, "metadata")

shutil.rmtree(OUT_ROOT, ignore_errors=True)
os.makedirs(META_DIR, exist_ok=True)

for task in ["sr", "colorization"]:
    for split in ["train", "val", "test"]:
        subfolders = ["tir_200m", "tir_100m"] if task == "sr" else ["tir_100m", "rgb_100m"]
        for sub in subfolders:
            os.makedirs(os.path.join(OUT_ROOT, task, split, sub), exist_ok=True)

SR_TIR100_SIZE = 512
SR_TIR200_SIZE = 256
COLOR_SIZE = 256

PATCHES_PER_SCENE = 30
MAX_ATTEMPTS_PER_SCENE = 5000

VALID_THRESHOLD = 0.95
MIN_RGB_STD = 0.005
MIN_TIR_STD = 0.10

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

scene_files = sorted(glob.glob(os.path.join(LOCAL_SCENE_DIR, "processed100m_*.tif")))
print("Scenes to process:", len(scene_files))


def get_scene_index_and_id(path):
    base = os.path.basename(path)
    m = re.match(r"processed100m_(\d{3})_(.+)\.tif$", base)
    if m:
        return int(m.group(1)), m.group(2)
    return -1, base.replace(".tif", "")


def split_for_rank(rank, total):
    train_cut = int(total * 0.70)
    val_cut = int(total * 0.85)
    if rank < train_cut:
        return "train"
    elif rank < val_cut:
        return "val"
    else:
        return "test"


def downsample_2x_mean(arr):
    if not np.isfinite(arr).all():
        return None
    return arr.reshape(SR_TIR200_SIZE, 2, SR_TIR200_SIZE, 2).mean(axis=(1, 3)).astype(np.float32)


def quality_ok(rgb512, tir512):
    valid_rgb = np.isfinite(rgb512).all(axis=0)
    valid_tir = np.isfinite(tir512)

    rgb_range_ok = (
        (rgb512[0] > 0) & (rgb512[0] <= 1) &
        (rgb512[1] > 0) & (rgb512[1] <= 1) &
        (rgb512[2] > 0) & (rgb512[2] <= 1)
    )

    tir_range_ok = (tir512 >= 250) & (tir512 <= 350)
    valid = valid_rgb & valid_tir & rgb_range_ok & tir_range_ok

    if valid.mean() < VALID_THRESHOLD:
        return False
    if not np.isfinite(rgb512).all():
        return False
    if not np.isfinite(tir512).all():
        return False

    rgb_vals = rgb512[:, valid]
    tir_vals = tir512[valid]

    if rgb_vals.size == 0 or tir_vals.size == 0:
        return False
    if np.nanstd(rgb_vals) < MIN_RGB_STD:
        return False
    if np.nanstd(tir_vals) < MIN_TIR_STD:
        return False

    return True


def center_crop_256(arr):
    start = (SR_TIR100_SIZE - COLOR_SIZE) // 2
    end = start + COLOR_SIZE
    if arr.ndim == 3:
        return arr[:, start:end, start:end]
    return arr[start:end, start:end]


metadata_rows = []
scene_summary_rows = []

for rank, scene_path in enumerate(scene_files):
    scene_index, product_id = get_scene_index_and_id(scene_path)
    split = split_for_rank(rank, len(scene_files))

    print("\nProcessing:", os.path.basename(scene_path))
    print("Split:", split)

    with rasterio.open(scene_path) as src:
        print("Bands:", src.count, "Shape:", src.height, src.width, "CRS:", src.crs)

        arr = src.read().astype(np.float32)

        if arr.shape[0] != 4:
            print("Skipping: expected 4 bands R,G,B,TIR")
            scene_summary_rows.append([scene_index, product_id, split, src.height, src.width, 0, "bad_band_count"])
            continue

        rgb = arr[:3]
        tir100_full = arr[3]
        H, W = tir100_full.shape

        if H < SR_TIR100_SIZE or W < SR_TIR100_SIZE:
            print("Skipping scene because too small:", H, W)
            scene_summary_rows.append([scene_index, product_id, split, H, W, 0, "too_small"])
            continue

        saved = 0
        attempts = 0

        while saved < PATCHES_PER_SCENE and attempts < MAX_ATTEMPTS_PER_SCENE:
            attempts += 1

            row = random.randint(0, H - SR_TIR100_SIZE)
            col = random.randint(0, W - SR_TIR100_SIZE)

            rgb512 = rgb[:, row:row + SR_TIR100_SIZE, col:col + SR_TIR100_SIZE]
            tir512 = tir100_full[row:row + SR_TIR100_SIZE, col:col + SR_TIR100_SIZE]

            if rgb512.shape != (3, SR_TIR100_SIZE, SR_TIR100_SIZE):
                continue
            if tir512.shape != (SR_TIR100_SIZE, SR_TIR100_SIZE):
                continue
            if not quality_ok(rgb512, tir512):
                continue

            sr_tir100 = tir512.astype(np.float32)
            sr_tir200 = downsample_2x_mean(sr_tir100)
            if sr_tir200 is None:
                continue
            if sr_tir200.shape != (SR_TIR200_SIZE, SR_TIR200_SIZE):
                continue
            if not np.isfinite(sr_tir200).all():
                continue

            color_tir100 = center_crop_256(sr_tir100).astype(np.float32)
            color_rgb100 = center_crop_256(rgb512).astype(np.float32)

            if color_tir100.shape != (COLOR_SIZE, COLOR_SIZE):
                continue
            if color_rgb100.shape != (3, COLOR_SIZE, COLOR_SIZE):
                continue

            patch_id = f"{split}_scene{scene_index:03d}_patch{saved:03d}"

            sr_tir200_rel = os.path.join("sr", split, "tir_200m", f"{patch_id}.npy")
            sr_tir100_rel = os.path.join("sr", split, "tir_100m", f"{patch_id}.npy")
            color_tir100_rel = os.path.join("colorization", split, "tir_100m", f"{patch_id}.npy")
            color_rgb100_rel = os.path.join("colorization", split, "rgb_100m", f"{patch_id}.npy")

            np.save(os.path.join(OUT_ROOT, sr_tir200_rel), sr_tir200)
            np.save(os.path.join(OUT_ROOT, sr_tir100_rel), sr_tir100)
            np.save(os.path.join(OUT_ROOT, color_tir100_rel), color_tir100)
            np.save(os.path.join(OUT_ROOT, color_rgb100_rel), color_rgb100)

            metadata_rows.append([
                patch_id, scene_index, product_id, split, row, col,
                sr_tir200_rel, sr_tir100_rel, color_tir100_rel, color_rgb100_rel
            ])

            saved += 1

        status = "ok" if saved > 0 else "no_good_patches"
        scene_summary_rows.append([scene_index, product_id, split, H, W, saved, status])
        print("Saved patches:", saved, "Attempts:", attempts)

metadata_path = os.path.join(META_DIR, "patch_metadata.csv")
scene_summary_path = os.path.join(META_DIR, "scene_summary.csv")

with open(metadata_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "patch_id", "scene_index", "product_id", "split", "row_100m", "col_100m",
        "sr_tir200_path", "sr_tir100_path", "color_tir100_path", "color_rgb100_path"
    ])
    writer.writerows(metadata_rows)

with open(scene_summary_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["scene_index", "product_id", "split", "height", "width", "saved_patches", "status"])
    writer.writerows(scene_summary_rows)

print("\nDONE")
print("Total paired samples:", len(metadata_rows))
print("Patch metadata:", metadata_path)
print("Scene summary:", scene_summary_path)

## 10. Count generated patches

In [ ]:
def count_generated_patches(root=OUT_ROOT):
    for task in ["sr", "colorization"]:
        print("\nTASK:", task)
        for split in ["train", "val", "test"]:
            print(" ", split)
            subfolders = ["tir_200m", "tir_100m"] if task == "sr" else ["tir_100m", "rgb_100m"]
            for sub in subfolders:
                folder = os.path.join(root, task, split, sub)
                print("   ", sub, len(os.listdir(folder)))

count_generated_patches()

## 11. Visual sanity check

Run this after patch extraction. Check train, val, and test samples.

In [ ]:
def show_sample(split="train", idx=0, root=OUT_ROOT):
    sr_tir200_dir = os.path.join(root, "sr", split, "tir_200m")
    files = sorted(os.listdir(sr_tir200_dir))

    if len(files) == 0:
        print("No files in", split)
        return

    idx = min(idx, len(files) - 1)
    sample_file = files[idx]
    patch_id = sample_file.replace(".npy", "")

    sr_tir200 = np.load(os.path.join(root, "sr", split, "tir_200m", sample_file))
    sr_tir100 = np.load(os.path.join(root, "sr", split, "tir_100m", sample_file))
    color_tir100 = np.load(os.path.join(root, "colorization", split, "tir_100m", sample_file))
    color_rgb100 = np.load(os.path.join(root, "colorization", split, "rgb_100m", sample_file))

    print("Patch:", patch_id)
    print("SR TIR200:", sr_tir200.shape, np.nanmin(sr_tir200), np.nanmax(sr_tir200), np.nanstd(sr_tir200))
    print("SR TIR100:", sr_tir100.shape, np.nanmin(sr_tir100), np.nanmax(sr_tir100), np.nanstd(sr_tir100))
    print("Color TIR100:", color_tir100.shape, np.nanmin(color_tir100), np.nanmax(color_tir100), np.nanstd(color_tir100))
    print("Color RGB100:", color_rgb100.shape, np.nanmin(color_rgb100), np.nanmax(color_rgb100), np.nanstd(color_rgb100))

    rgb_disp = np.moveaxis(color_rgb100, 0, -1)
    valid = np.isfinite(rgb_disp).all(axis=2)
    vals = rgb_disp[valid]

    p2, p98 = np.percentile(vals, [2, 98])
    rgb_disp = np.clip((rgb_disp - p2) / (p98 - p2), 0, 1)

    fig, ax = plt.subplots(1, 4, figsize=(18, 5))

    ax[0].imshow(sr_tir200, cmap="gray")
    ax[0].set_title("SR Input\nTIR 200m 256×256")
    ax[0].axis("off")

    ax[1].imshow(sr_tir100, cmap="gray")
    ax[1].set_title("SR Target\nTIR 100m 512×512")
    ax[1].axis("off")

    ax[2].imshow(color_tir100, cmap="gray")
    ax[2].set_title("Color Input\nTIR 100m 256×256")
    ax[2].axis("off")

    ax[3].imshow(rgb_disp)
    ax[3].set_title("Color Target\nRGB 100m 256×256")
    ax[3].axis("off")

    plt.tight_layout()
    plt.show()

show_sample("train", 0)
# show_sample("val", 0)
# show_sample("test", 0)

## 12. Save dataset to Drive and create ZIP

Run this after the counts and visual checks are good.

In [ ]:
DRIVE_DATASET_OUT = os.path.join(DRIVE_ROOT, "dataset_current_repo_format")

shutil.rmtree(DRIVE_DATASET_OUT, ignore_errors=True)
shutil.copytree(OUT_ROOT, DRIVE_DATASET_OUT)

print("Copied dataset to:", DRIVE_DATASET_OUT)

zip_base = DRIVE_DATASET_OUT
shutil.make_archive(zip_base, "zip", DRIVE_DATASET_OUT)
print("ZIP created:", zip_base + ".zip")

## 13. PyTorch dataloaders

Use these for model development.

Expected shapes:

```text
SR:
x = [B, 1, 256, 256]
y = [B, 1, 512, 512]

Colorization:
x = [B, 1, 256, 256]
y = [B, 3, 256, 256]
```

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

DATASET_ROOT = OUT_ROOT

class SRDataset(Dataset):
    def __init__(self, root, split="train"):
        self.tir200_dir = os.path.join(root, "sr", split, "tir_200m")
        self.tir100_dir = os.path.join(root, "sr", split, "tir_100m")
        self.files = sorted(glob.glob(os.path.join(self.tir200_dir, "*.npy")))
        print(f"SR {split} samples:", len(self.files))

    def __len__(self):
        return len(self.files)

    def normalize_tir(self, x):
        x = (x - 250.0) / 100.0
        return np.clip(x, 0, 1)

    def __getitem__(self, idx):
        tir200_path = self.files[idx]
        name = os.path.basename(tir200_path)
        tir100_path = os.path.join(self.tir100_dir, name)

        x = np.load(tir200_path).astype(np.float32)
        y = np.load(tir100_path).astype(np.float32)

        x = self.normalize_tir(x)
        y = self.normalize_tir(y)

        x = torch.from_numpy(x).unsqueeze(0)
        y = torch.from_numpy(y).unsqueeze(0)

        return x, y, name


class ColorizationDataset(Dataset):
    def __init__(self, root, split="train"):
        self.tir_dir = os.path.join(root, "colorization", split, "tir_100m")
        self.rgb_dir = os.path.join(root, "colorization", split, "rgb_100m")
        self.files = sorted(glob.glob(os.path.join(self.tir_dir, "*.npy")))
        print(f"Colorization {split} samples:", len(self.files))

    def __len__(self):
        return len(self.files)

    def normalize_tir(self, x):
        x = (x - 250.0) / 100.0
        return np.clip(x, 0, 1)

    def __getitem__(self, idx):
        tir_path = self.files[idx]
        name = os.path.basename(tir_path)
        rgb_path = os.path.join(self.rgb_dir, name)

        x = np.load(tir_path).astype(np.float32)
        y = np.load(rgb_path).astype(np.float32)

        x = self.normalize_tir(x)
        y = np.clip(y, 0, 1)

        x = torch.from_numpy(x).unsqueeze(0)
        y = torch.from_numpy(y)

        return x, y, name

BATCH_SIZE = 4

sr_train = SRDataset(DATASET_ROOT, "train")
sr_loader = DataLoader(sr_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

color_train = ColorizationDataset(DATASET_ROOT, "train")
color_loader = DataLoader(color_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

x_sr, y_sr, names_sr = next(iter(sr_loader))
x_col, y_col, names_col = next(iter(color_loader))

print("SR input:", x_sr.shape)
print("SR target:", y_sr.shape)
print("Color input:", x_col.shape)
print("Color target:", y_col.shape)

## 14. Continuing in another notebook

Yes, you can continue in another notebook.

Only these are required:

1. Mount Google Drive.
2. Set `DATASET_ROOT` to:

```python
DATASET_ROOT = "/content/drive/MyDrive/landsat_india_200/dataset_current_repo_format"
```

3. Reuse the PyTorch dataloader classes from section 13.

Do not depend on `/content/landsat_india_200_work` in another notebook unless you copy files there again, because `/content/` is temporary.

## 15. GitHub push plan

Do **not** push the full dataset `.npy` or `.tif` files to GitHub.

Recommended repo structure:

```text
IR-colorization-BAH2026-work/
  notebooks/
    dataset_pipeline.ipynb
    dataloader_sanity_check.ipynb

  src/
    data/
      datasets.py
    models/
      sr_model.py
      colorization_model.py
    train_sr.py
    train_colorization.py
    utils.py

  docs/
    TIR_SR_Colorization_Dataset_Documentation.md

  .gitignore
  README.md
  requirements.txt
```

Recommended `.gitignore`:

```gitignore
# data outputs
*.npy
*.tif
*.tiff
*.zip
*.pth
*.pt
*.ckpt

# folders
dataset/
dataset_current/
dataset_current_repo_format/
landsat_india_200/
outputs/
checkpoints/
runs/
wandb/

# Python
__pycache__/
.ipynb_checkpoints/
.env
```

Only push:
- notebooks
- source code
- metadata examples
- documentation
- small sample images if needed

Keep the actual dataset in Google Drive or another external storage location.